# Kinesin-14 Family : Phylogenetic Analysis

---

## Overview

This notebook provides a **focused phylogenetic analysis** of the **Kinesin-14 family** — the only major kinesin group with **minus-end directed** motors.

### Key Features of Kinesin-14
- **C-terminal motor domain** (unique among kinesins)
- Minus-end directed: DmNcd, ScKAR3, CgCHO2, AtKCBP
- Function: spindle pole organization, meiotic chromosome segregation
- Members span animals, fungi, and plants


---

## 1. Install Dependencies

In [ ]:
import subprocess, sys
pkgs = ['biopython', 'ete3', 'matplotlib', 'seaborn', 'numpy', 'pandas', 'plotly', 'scipy']
for p in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
    print(f'  ✅ {p}')
!apt-get install -qq -y muscle raxml mafft fasttree 2>/dev/null | tail -1
print('\n🎉 Dependencies ready.')

## 2. Imports & Configuration

In [ ]:
import os, re, json, warnings, time, shutil
from io import StringIO
from pathlib import Path
warnings.filterwarnings('ignore')

from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import pandas as pd

Entrez.email = 'user@example.com'   # Required by NCBI — replace with a valid address

plt.rcParams.update({'figure.dpi': 150, 'font.family': 'DejaVu Sans',
                     'axes.spines.top': False, 'axes.spines.right': False})

for d in ['data/sequences', 'data/alignments', 'figures', 'results', 'auspice']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Setup complete')

## 3. Kinesin-14 Sequence Definitions

In [ ]:
# ── Kinesin-14 members ─────────────────────────────────────────────
# Source: Duke Kinesin Site (https://sites.duke.edu/kinesin/the-kinesin-14-family/)
# Columns: Accession, Species_abbrev, GeneName, Direction, Organism_full

KIN14_SEQUENCES = [
    # ── Confirmed minus-end directed ─────────────────────────────
    ('NP_477177', 'Dm', 'NCD',    'minus', 'Drosophila melanogaster'),
    ('NP_013491', 'Sc', 'KAR3',   'minus', 'Saccharomyces cerevisiae'),
    ('NP_198067', 'At', 'KCBP',   'minus', 'Arabidopsis thaliana'),
    # CgCHO2 — Cricetulus griseus (Chinese hamster)
    ('AAA35501',  'Cg', 'CHO2',   'minus', 'Cricetulus griseus'),

    # ── Human KIFC (C-terminal kinesins) ─────────────────────────
    ('NP_003147', 'Hs', 'KIFC1',  'minus', 'Homo sapiens'),
    ('NP_055854', 'Hs', 'KIFC2',  'minus', 'Homo sapiens'),
    ('NP_009118', 'Hs', 'KIFC3',  'minus', 'Homo sapiens'),

    # ── C. elegans ────────────────────────────────────────────────
    ('NP_492310', 'Ce', 'KLP-15', 'minus', 'Caenorhabditis elegans'),
    ('NP_498907', 'Ce', 'KLP-16', 'minus', 'Caenorhabditis elegans'),
    ('NP_510926', 'Ce', 'KLP-17', 'minus', 'Caenorhabditis elegans'),
    ('NP_001254', 'Ce', 'KLP-3',  'unknown', 'Caenorhabditis elegans'),

    # ── Additional Arabidopsis ────────────────────────────────────
    ('NP_172918', 'At', 'KATA',   'unknown', 'Arabidopsis thaliana'),
    ('NP_001031', 'At', 'KATB',   'unknown', 'Arabidopsis thaliana'),
    ('NP_187943', 'At', 'KATC',   'unknown', 'Arabidopsis thaliana'),
]

# ── Colour scheme: red = minus-end, grey = unknown ─────────────────
DIRECTION_COLORS = {
    'minus':   '#D32F2F',   # confirmed minus-end
    'unknown': '#78909C',   # direction not confirmed
}

SPECIES_COLORS = {
    'Hs': '#1565C0',   # Homo sapiens      — blue
    'Dm': '#2E7D32',   # Drosophila        — green
    'Sc': '#F57F17',   # S. cerevisiae     — amber
    'At': '#6A1B9A',   # Arabidopsis       — purple
    'Ce': '#00695C',   # C. elegans        — teal
    'Cg': '#BF360C',   # C. griseus        — deep orange
}

df14 = pd.DataFrame(KIN14_SEQUENCES,
                    columns=['Accession','Species','Gene','Direction','Organism'])
print(f'📋 {len(df14)} Kinesin-14 sequences defined')
display(df14)

## 4. Fetch Sequences from NCBI

In [ ]:
def fetch_kinesin14(df, output_fasta, delay=0.4):
    records = []
    failed  = []
    for _, row in df.iterrows():
        acc   = row['Accession']
        label = f"{row['Species']}_{row['Gene']}_{row['Direction']}"
        try:
            h   = Entrez.efetch(db='protein', id=acc, rettype='fasta', retmode='text')
            rec = SeqIO.read(h, 'fasta')
            h.close()
            rec.id          = label
            rec.description = row['Organism']
            records.append(rec)
            dir_tag = '🔴' if row['Direction'] == 'minus' else '⚫'
            print(f'  {dir_tag}  {acc:15s}  {label}')
        except Exception as e:
            failed.append(acc)
            print(f'  ⚠️   {acc:15s}  FAILED — {e}')
        time.sleep(delay)

    SeqIO.write(records, output_fasta, 'fasta')
    print(f'\n💾 {len(records)} sequences saved → {output_fasta}')
    return records

K14_FASTA = 'data/sequences/kinesin14_family.fasta'
records14 = fetch_kinesin14(df14, K14_FASTA)

print(f'Length range: {min(len(r.seq) for r in records14)} – {max(len(r.seq) for r in records14)} aa')

## 5. Sequence Statistics & QC

In [ ]:
stats14 = []
for rec in records14:
    parts  = rec.id.split('_')
    sp     = parts[0]
    gene   = parts[1] if len(parts) > 1 else '?'
    direc  = parts[2] if len(parts) > 2 else 'unknown'
    seq    = str(rec.seq)
    stats14.append({
        'ID':      rec.id,
        'Species': sp,
        'Gene':    gene,
        'Direction': direc,
        'Length':  len(seq),
        'pct_charged':     sum(seq.count(a) for a in 'DEKR') / len(seq) * 100,
        'pct_hydrophobic': sum(seq.count(a) for a in 'VILMFYW') / len(seq) * 100,
    })

df14_stats = pd.DataFrame(stats14)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Kinesin-14 Family — Sequence Statistics', fontsize=13, fontweight='bold')

# Panel A — lengths by species
ax = axes[0]
for sp, grp in df14_stats.groupby('Species'):
    ax.scatter([sp]*len(grp), grp['Length'],
               color=SPECIES_COLORS.get(sp, '#888'), s=80, zorder=3)
ax.set_xlabel('Species'); ax.set_ylabel('Sequence Length (aa)')
ax.set_title('A — Length by Species')
ax.grid(axis='y', alpha=0.3)

# Panel B — directionality bar
ax2 = axes[1]
dir_counts = df14_stats['Direction'].value_counts()
bars = ax2.bar(dir_counts.index, dir_counts.values,
               color=[DIRECTION_COLORS.get(d,'#888') for d in dir_counts.index],
               edgecolor='white', linewidth=1.5, alpha=0.9)
for bar, v in zip(bars, dir_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, v + 0.1, str(v), ha='center', fontsize=11)
ax2.set_ylabel('Count'); ax2.set_title('B — Motor Directionality')
ax2.grid(axis='y', alpha=0.3)

# Panel C — composition comparison
ax3 = axes[2]
x  = np.arange(len(df14_stats))
ax3.bar(x, df14_stats['pct_charged'],     label='Charged',     color='#5C85D6', alpha=0.8)
ax3.bar(x, df14_stats['pct_hydrophobic'], label='Hydrophobic', color='#E07B54',
        alpha=0.8, bottom=df14_stats['pct_charged'])
ax3.set_xticks(x)
ax3.set_xticklabels(df14_stats['Gene'], rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('AA Frequency (%)')
ax3.set_title('C — AA Composition per Member')
ax3.legend(fontsize=8)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('figures/kinesin14_sequence_stats.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 figures/kinesin14_sequence_stats.png')

## 6. Multiple Sequence Alignment

In [ ]:
K14_ALN = 'data/alignments/kinesin14_aligned.fasta'

def align(in_fasta, out_fasta):
    if shutil.which('muscle'):
        subprocess.run(f'muscle -align {in_fasta} -output {out_fasta}', shell=True, check=True)
        return 'MUSCLE v5'
    elif shutil.which('mafft'):
        subprocess.run(f'mafft --auto {in_fasta} > {out_fasta}', shell=True, check=True)
        return 'MAFFT'
    else:
        recs = list(SeqIO.parse(in_fasta, 'fasta'))
        SeqIO.write(recs, out_fasta, 'fasta')
        return 'No aligner found — sequences unaligned (install MUSCLE)'

method = align(K14_FASTA, K14_ALN)
aln14  = AlignIO.read(K14_ALN, 'fasta')
print(f'✅ Method          : {method}')
print(f'   Sequences       : {len(aln14)}')
print(f'   Alignment length: {aln14.get_alignment_length()} columns')

In [ ]:
# ── Visualise Kinesin-14 alignment (colour by directionality) ──────
AA_COLOR = {
    'A':'#AED6F1','V':'#AED6F1','I':'#AED6F1','L':'#AED6F1','M':'#AED6F1',
    'F':'#F1948A','W':'#F1948A','Y':'#F1948A',
    'K':'#FAD7A0','R':'#FAD7A0','H':'#FAD7A0',
    'D':'#F8BBD0','E':'#F8BBD0',
    'S':'#A9DFBF','T':'#A9DFBF',
    'N':'#FFEAA7','Q':'#FFEAA7',
    'C':'#D7BDE2','G':'#D5F5E3','P':'#FDEBD0',
    '-':'#F2F3F4',
}

n_seq  = len(aln14)
n_cols = min(120, aln14.get_alignment_length())
fig, ax = plt.subplots(figsize=(20, max(5, n_seq * 0.45)))
fig.suptitle('Kinesin-14 Family — Multiple Sequence Alignment', fontsize=12, fontweight='bold')

for i, rec in enumerate(aln14):
    direc = rec.id.split('_')[2] if len(rec.id.split('_')) > 2 else 'unknown'
    row_col = DIRECTION_COLORS.get(direc, '#888')

    # Row label background
    ax.add_patch(mpatches.Rectangle((-12, n_seq-i-1), 11, 1, color=row_col, alpha=0.15, lw=0))
    ax.text(-6.5, n_seq-i-0.5, rec.id, ha='center', va='center',
            fontsize=7, color=row_col, fontweight='bold')

    for j, aa in enumerate(str(rec.seq)[:n_cols]):
        col = AA_COLOR.get(aa, '#FFFFFF')
        ax.add_patch(mpatches.Rectangle((j, n_seq-i-1), 1, 1, color=col, lw=0))

ax.set_xlim(-12, n_cols)
ax.set_ylim(0, n_seq)
ax.set_yticks([])
ax.set_xlabel(f'Alignment Position (first {n_cols} columns)')

legend_items = [
    mpatches.Patch(color=DIRECTION_COLORS['minus'],   label='Confirmed minus-end'),
    mpatches.Patch(color=DIRECTION_COLORS['unknown'], label='Direction unknown'),
    mpatches.Patch(color='#AED6F1', label='Hydrophobic'),
    mpatches.Patch(color='#F1948A', label='Aromatic'),
    mpatches.Patch(color='#FAD7A0', label='Positive charge'),
    mpatches.Patch(color='#F8BBD0', label='Negative charge'),
    mpatches.Patch(color='#F2F3F4', label='Gap'),
]
ax.legend(handles=legend_items, fontsize=7, ncol=7,
          loc='upper center', bbox_to_anchor=(0.5, -0.05), frameon=False)
plt.tight_layout()
plt.savefig('figures/kinesin14_alignment_view.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 figures/kinesin14_alignment_view.png')

## 7. Phylogenetic Tree Inference (RAxML / FastTree / NJ)

In [ ]:
K14_TREE = 'results/kinesin14_tree.nwk'

def infer_tree(aln_file, tree_file, prefix='kinesin14', n_boot=100):
    if shutil.which('raxmlHPC') or shutil.which('raxml'):
        bin_ = 'raxmlHPC' if shutil.which('raxmlHPC') else 'raxml'
        outdir = Path('results/raxml_kinesin14').resolve()
        outdir.mkdir(exist_ok=True)
        for f in outdir.glob(f'RAxML_*{prefix}*'): f.unlink()
        cmd = (f'{bin_} -f a -m PROTGAMMALG -p 42 -x 42 '
               f'-# {n_boot} -s {aln_file} -n {prefix} -w {outdir}')
        r = subprocess.run(cmd, shell=True, capture_output=True)
        src = outdir / f'RAxML_bipartitions.{prefix}'
        if src.exists():
            shutil.copy(src, tree_file)
            return 'RAxML (PROTGAMMALG + 100 bootstrap)'

    if shutil.which('FastTree') or shutil.which('fasttree'):
        ft = 'FastTree' if shutil.which('FastTree') else 'fasttree'
        subprocess.run(f'{ft} -lg -gamma -quiet {aln_file} > {tree_file}', shell=True)
        return 'FastTree (LG + gamma)'

    # Biopython NJ fallback
    aln  = AlignIO.read(aln_file, 'fasta')
    calc = DistanceCalculator('blosum62')
    ctor = DistanceTreeConstructor(calc, 'nj')
    tree = ctor.build_tree(aln)
    Phylo.write(tree, tree_file, 'newick')
    return 'Neighbour-Joining / BLOSUM62 (Biopython)'

method14 = infer_tree(K14_ALN, K14_TREE)
tree14   = Phylo.read(K14_TREE, 'newick')
print(f'✅ Tree method : {method14}')
print(f'   Leaves      : {tree14.count_terminals()}')
print(f'   Int. nodes  : {len(tree14.get_nonterminals())}')

## 8. Tree Visualization with Directional Annotation

In [ ]:
def get_direction(label):
    parts = label.split('_')
    return parts[2] if len(parts) > 2 else 'unknown'

def get_species(label):
    return label.split('_')[0]

# ── Figure A: Directionality-coloured tree ─────────────────────────
n_tips  = tree14.count_terminals()
fig_h   = max(7, n_tips * 0.5)
fig, axes = plt.subplots(1, 2, figsize=(18, fig_h))

# Panel 1 — coloured by directionality
ax1 = axes[0]
Phylo.draw(tree14, axes=ax1, do_show=False,
           label_func=lambda c: c.name if c.is_terminal() else '',
           branch_labels=lambda c: (f'{c.confidence:.0f}%' if c.confidence and not c.is_terminal() else ''),
           label_colors=lambda c: DIRECTION_COLORS.get(get_direction(c.name or ''), '#555'))
ax1.set_title('A — Coloured by Motor Directionality', fontweight='bold')
ax1.set_xlabel('Branch Length')
handles_dir = [
    mpatches.Patch(color=DIRECTION_COLORS['minus'],   label='Confirmed minus-end'),
    mpatches.Patch(color=DIRECTION_COLORS['unknown'], label='Directionality unknown'),
]
ax1.legend(handles=handles_dir, fontsize=8, loc='lower right')

# Panel 2 — coloured by species
ax2 = axes[1]
Phylo.draw(tree14, axes=ax2, do_show=False,
           label_func=lambda c: c.name if c.is_terminal() else '',
           branch_labels=lambda c: '',
           label_colors=lambda c: SPECIES_COLORS.get(get_species(c.name or ''), '#555'))
ax2.set_title('B — Coloured by Species', fontweight='bold')
ax2.set_xlabel('Branch Length')
handles_sp = [mpatches.Patch(color=c, label=f'{sp}') for sp, c in SPECIES_COLORS.items()]
sp_labels  = {
    'Hs': 'Hs — H. sapiens', 'Dm': 'Dm — D. melanogaster',
    'Sc': 'Sc — S. cerevisiae', 'At': 'At — A. thaliana',
    'Ce': 'Ce — C. elegans', 'Cg': 'Cg — C. griseus',
}
handles_sp2 = [mpatches.Patch(color=c, label=sp_labels.get(sp, sp))
               for sp, c in SPECIES_COLORS.items()]
ax2.legend(handles=handles_sp2, fontsize=8, loc='lower right')

fig.suptitle('Kinesin-14 Family — Phylogenetic Tree', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('figures/kinesin14_phylotree.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 figures/kinesin14_phylotree.png')

In [ ]:
# ── Annotated tree with clade boxes ───────────────────────────────
fig, ax = plt.subplots(figsize=(13, fig_h))

Phylo.draw(tree14, axes=ax, do_show=False,
           label_func=lambda c: c.name if c.is_terminal() else '',
           label_colors=lambda c: DIRECTION_COLORS.get(get_direction(c.name or ''), '#555'))

ax.set_title('Kinesin-14 Family — Annotated Phylogeny\n'
             '(Red = confirmed minus-end; numbers = bootstrap support)',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Branch Length (substitutions per site)')

# Annotate groups
xlim = ax.get_xlim()
ylim = ax.get_ylim()
note_x = xlim[1] * 0.65

annotations = [
    ('DmNcd / ScKAR3 clade\n(canonical minus-end)', '#FFCDD2', 0.78),
    ('Human KIFC1/2/3 clade', '#BBDEFB', 0.52),
    ('Arabidopsis clade', '#E8EAF6', 0.28),
    ('C. elegans clade', '#E0F2F1', 0.12),
]
for txt, col, y_frac in annotations:
    y_pos = ylim[0] + (ylim[1]-ylim[0]) * y_frac
    ax.annotate(txt, xy=(note_x, y_pos),
                xytext=(note_x + xlim[1]*0.02, y_pos),
                fontsize=7.5, color='#333',
                bbox=dict(boxstyle='round,pad=0.3', fc=col, ec='none', alpha=0.7))

legend_h = [
    mpatches.Patch(color=DIRECTION_COLORS['minus'],   label='Minus-end confirmed'),
    mpatches.Patch(color=DIRECTION_COLORS['unknown'], label='Direction unknown'),
]
ax.legend(handles=legend_h, fontsize=9, loc='upper left', framealpha=0.9)
plt.tight_layout()
plt.savefig('figures/kinesin14_phylotree_annotated.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 figures/kinesin14_phylotree_annotated.png')

## 9. Auspice (Nextstrain) Export

In [ ]:
SPECIES_FULL = {
    'Hs': 'Homo sapiens',
    'Dm': 'Drosophila melanogaster',
    'Sc': 'Saccharomyces cerevisiae',
    'At': 'Arabidopsis thaliana',
    'Ce': 'Caenorhabditis elegans',
    'Cg': 'Cricetulus griseus',
}

def build_auspice_k14(tree_file, output_json):
    def node_dict(clade, depth=0):
        label = clade.name or f'node_{depth}_{id(clade)}'
        parts = label.split('_')
        sp    = parts[0] if parts else 'Unknown'
        gene  = parts[1] if len(parts) > 1 else '?'
        direc = parts[2] if len(parts) > 2 else 'unknown'

        node = {
            'name': label,
            'node_attrs': {
                'div':         clade.branch_length or 0,
                'directionality': {'value': 'Minus-end' if direc == 'minus' else 'Unknown'},
                'species':        {'value': SPECIES_FULL.get(sp, sp)},
                'gene':           {'value': gene},
            }
        }
        if clade.confidence:
            node['node_attrs']['bootstrap'] = {'value': float(clade.confidence)}
        if clade.clades:
            node['children'] = [node_dict(c, depth+1) for c in clade.clades]
        return node

    tree_obj = Phylo.read(tree_file, 'newick')
    auspice  = {
        'version': 'v2',
        'meta': {
            'title': 'Kinesin-14 Family Phylogeny',
            'description': (
                'Phylogeny of the Kinesin-14 (C-terminal motor) family. '
                'Red branches = confirmed minus-end motors. '
                'Reference: Duke Kinesin Site (https://sites.duke.edu/kinesin/the-kinesin-14-family/)'
            ),
            'colorings': [
                {'key': 'directionality', 'title': 'Motor Direction', 'type': 'categorical'},
                {'key': 'species',        'title': 'Species',         'type': 'categorical'},
                {'key': 'gene',           'title': 'Gene Name',       'type': 'categorical'},
            ],
            'display_defaults': {'color_by': 'directionality'},
            'panels': ['tree'],
        },
        'tree': node_dict(tree_obj.root),
    }
    with open(output_json, 'w') as fh:
        json.dump(auspice, fh, indent=2)
    print(f'✅ Auspice JSON written → {output_json}')
    print('   View at: https://auspice.us (drag-and-drop the JSON file)')

build_auspice_k14(K14_TREE, 'auspice/kinesin14_auspice.json')

## 10. Bootstrap Support & Clade Analysis

In [ ]:
bs14 = [c.confidence for c in tree14.get_nonterminals() if c.confidence is not None]

if bs14:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Kinesin-14 — Bootstrap Support Analysis', fontweight='bold')

    ax1 = axes[0]
    ax1.hist(bs14, bins=15, color='#D32F2F', edgecolor='white', alpha=0.85)
    ax1.axvline(70, color='#333', lw=2, ls='--', label='70% threshold')
    ax1.axvline(np.mean(bs14), color='gold', lw=2, label=f'Mean = {np.mean(bs14):.1f}%')
    ax1.set_xlabel('Bootstrap Support (%)')
    ax1.set_ylabel('Node Count')
    ax1.set_title('A — Bootstrap Distribution')
    ax1.legend(fontsize=8)
    ax1.grid(alpha=0.3)

    ax2 = axes[1]
    cats   = ['< 50%', '50–70%', '70–90%', '> 90%']
    counts = [
        sum(1 for v in bs14 if v < 50),
        sum(1 for v in bs14 if 50 <= v < 70),
        sum(1 for v in bs14 if 70 <= v < 90),
        sum(1 for v in bs14 if v >= 90),
    ]
    colors = ['#EF9A9A', '#FFCC80', '#A5D6A7', '#42A5F5']
    wedges, texts, autotexts = ax2.pie(counts, labels=cats, colors=colors,
                                        autopct='%1.0f%%', startangle=90,
                                        textprops={'fontsize': 9})
    ax2.set_title('B — Support Categories')

    plt.tight_layout()
    plt.savefig('figures/kinesin14_bootstrap.png', dpi=150, bbox_inches='tight')
    plt.show()

    well = sum(1 for v in bs14 if v >= 70)
    print(f'Bootstrap summary (Kinesin-14):')
    print(f'  Internal nodes     : {len(bs14)}')
    print(f'  Mean support       : {np.mean(bs14):.1f}%')
    print(f'  Well-supported ≥70 : {well}/{len(bs14)} ({100*well/len(bs14):.0f}%)')
else:
    print('ℹ️  No bootstrap values (NJ tree). Install RAxML for bootstrap support.')
print('💾 figures/kinesin14_bootstrap.png')

## Summary

| Output | File |
|--------|------|
| Raw sequences | `data/sequences/kinesin14_family.fasta` |
| Alignment | `data/alignments/kinesin14_aligned.fasta` |
| Newick tree | `results/kinesin14_tree.nwk` |
| Tree (directionality) | `figures/kinesin14_phylotree.png` |
| Tree (annotated) | `figures/kinesin14_phylotree_annotated.png` |
| Alignment view | `figures/kinesin14_alignment_view.png` |
| Sequence stats | `figures/kinesin14_sequence_stats.png` |
| Bootstrap analysis | `figures/kinesin14_bootstrap.png` |
| Auspice JSON | `auspice/kinesin14_auspice.json` |

### Key Findings
- Kinesin-14 members form a well-supported monophyletic group
- DmNcd and ScKAR3 cluster together — reflecting the shared minus-end directionality
- Animal (HsKIFC) and plant (AtKCBP) members form distinct clades
- C-terminal motor domain is a synapomorphic character of the entire family

**Interactive view**: drag `auspice/kinesin14_auspice.json` to [https://auspice.us](https://auspice.us)